
# 02 - Manual Business Queries

## Goal

Validate the business SQL queries that will later become safe Python functions.

In this project, the foundation model will NOT generate SQL directly.

Instead, the system will work like this:

Business question  
↓  
Manual SQL query  
↓  
Validated result  
↓  
Safe Python function  
↓  
Foundation model selects the function later

## Example

User question:

¿Cuántas F150 vendí este mes?

↓

Manual SQL:

SELECT SUM(quantity), SUM(sale_price * quantity)
FROM auto_sales_transactions
WHERE product_model = 'F150'
AND sale_date BETWEEN ...

↓

Validated result:

5 F150 sold this month with $266,500 total revenue.

↓

Future function:

get_product_sales_summary(product_model="F150", date_range="this_month")

## Current step

We are here:

Business question  
↓  
SQL manual  
↓  
Validated result  
↓  
Later this query becomes a function

## Output of this notebook

A list of validated SQL queries that will become the function catalog in the next notebook.


In [0]:
# Read source table

SOURCE_TABLE = "auto_sales_transactions"

sales_df = spark.table(SOURCE_TABLE)

display(sales_df)

sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type,created_at
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance,2026-05-23T21:11:25.384Z
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease,2026-05-23T21:11:25.384Z
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance,2026-05-23T21:11:25.384Z
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash,2026-05-23T21:11:25.384Z
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance,2026-05-23T21:11:25.384Z
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance,2026-05-23T21:11:25.384Z
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease,2026-05-23T21:11:25.384Z
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance,2026-05-23T21:11:25.384Z
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash,2026-05-23T21:11:25.384Z
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance,2026-05-23T21:11:25.384Z


In [0]:
# Validate table exists and is populated

record_count = sales_df.count()

print(f"Records available in {SOURCE_TABLE}: {record_count}")

if record_count == 0:
    raise ValueError("Source table is empty. Please run Notebook 01 first.")

Records available in auto_sales_transactions: 15


In [0]:
%sql
-- Date range available
SELECT
  MIN(sale_date) AS min_sale_date,
  MAX(sale_date) AS max_sale_date,
  COUNT(*) AS total_records
FROM auto_sales_transactions;

min_sale_date,max_sale_date,total_records
2026-01-10,2026-05-20,15



## Query 1 - Product Sales Summary

### Business question

¿Cuántas F150 vendí este mes?

### Manual SQL

We filter by:

- product_model = 'F150'
- sale_date between 2026-05-01 and 2026-05-23

### Later this becomes

get_product_sales_summary(product_model, date_range)

### Why this matters

This function prevents the model from inventing sales numbers.  
The model will only select the function and arguments.  
- The SQL result comes from the Delta table.

In [0]:
%sql
-- Query F150 sales for this month
SELECT
  product_model,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue,
  AVG(sale_price) AS average_sale_price
FROM auto_sales_transactions
WHERE product_model = 'F150'
  AND sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY product_model;

product_model,units_sold,total_revenue,average_sale_price
F150,5,266500.0,53300.0



## Query 2 - Top Selling Models

### Business question

¿Cuáles son mis modelos más vendidos este mes?

### Manual SQL

We group sales by product_model and calculate:

- units_sold
- total_revenue

### Later this becomes

get_top_selling_models(date_range, limit)

### Why this matters

This lets the assistant answer ranking questions using actual sales data instead of model-generated guesses.

In [0]:
%sql
-- Top selling models this month
SELECT
  product_model,
  brand,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY product_model, brand
ORDER BY units_sold DESC, total_revenue DESC
LIMIT 5;

product_model,brand,units_sold,total_revenue
F150,Ford,5,266500.0
Silverado,Chevrolet,2,99500.0
Explorer,Ford,1,47000.0
Mustang,Ford,1,44000.0
RAV4,Toyota,1,39000.0



## Query 3 - Sales by Representative

### Business question

¿Qué vendedor vendió más este mes?

### Manual SQL

We group by sales_rep and calculate:

- units_sold
- total_revenue

### Later this becomes

get_sales_by_rep(date_range)

### Why this matters

This query supports management questions about sales performance.

In [0]:
%sql
-- Query sales by rep
SELECT
  sales_rep,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY sales_rep
ORDER BY units_sold DESC, total_revenue DESC;


## Query 4 - Revenue by State

### Business question

¿Cuánto vendimos por estado este mes?

### Manual SQL

We group by customer_state and calculate total revenue.

### Later this becomes

get_revenue_by_state(date_range)

### Why this matters

This helps answer regional performance questions using controlled SQL.

In [0]:
%sql
-- Query Revenue by state
SELECT
  customer_state,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY customer_state
ORDER BY total_revenue DESC;

customer_state,units_sold,total_revenue
TX,5,266500.0
CA,3,130500.0
AZ,2,91000.0
WA,1,39000.0
NV,1,28000.0



## Query 5 - Average Sale Price

### Business question

¿Cuál fue el ticket promedio de Mustang este mes?

### Manual SQL

We filter by product_model and calculate AVG(sale_price).

### Later this becomes

get_average_sale_price(product_model, date_range)

### Why this matters

This supports pricing and performance questions for a specific model.

In [0]:
%sql
-- Query Average sale price by model
SELECT
  product_model,
  COUNT(*) AS total_sales,
  AVG(sale_price) AS average_sale_price,
  MIN(sale_price) AS min_sale_price,
  MAX(sale_price) AS max_sale_price
FROM auto_sales_transactions
WHERE product_model = 'Mustang'
  AND sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY product_model;

product_model,total_sales,average_sale_price,min_sale_price,max_sale_price
Mustang,1,44000.0,44000.0,44000.0



## Query 6 - Dealership Sales Summary

### Business question

Dame ventas por dealership este mes.

### Manual SQL

We group by dealership_name and calculate:

- units_sold
- total_revenue
- average_sale_price

### Later this becomes

get_dealership_sales_summary(date_range)

### Why this matters

This helps compare performance across dealership locations.

In [0]:
%sql
-- Query Sales by dealership summary
SELECT
  dealership_name,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue,
  AVG(sale_price) AS average_sale_price
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY dealership_name
ORDER BY total_revenue DESC;

dealership_name,units_sold,total_revenue,average_sale_price
Acme Motors North,5,266500.0,53300.0
Acme Motors South,3,130500.0,43500.0
Acme Motors East,3,119000.0,39666.666666666664
Acme Motors West,1,39000.0,39000.0



# Function Candidates

After validating the manual SQL queries, we can define the safe functions that the assistant will be allowed to call.

The model will not generate SQL.

The model will only return JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Then Python will execute the corresponding validated function.

## Validated functions

| Function name | Business question | Main arguments |
|---|---|---|
| get_product_sales_summary | How many units and revenue for one model? | product_model, date_range |
| get_top_selling_models | What are the top selling models? | date_range, limit |
| get_sales_by_rep | Which sales rep sold the most? | date_range |
| get_revenue_by_state | How much revenue by state? | date_range |
| get_average_sale_price | What is the average sale price for a model? | product_model, date_range |
| get_dealership_sales_summary | How did each dealership perform? | date_range |

## Next notebook

In Notebook 03, we will create the function catalog.

That catalog will describe:

- function names
- descriptions
- allowed arguments
- example user questions
- expected JSON output

In [0]:
validated_functions = [
    {
        "function_name": "get_product_sales_summary",
        "business_question": "How many units and revenue for one vehicle model?",
        "arguments": ["product_model", "date_range"],
        "source_table": SOURCE_TABLE
    },
    {
        "function_name": "get_top_selling_models",
        "business_question": "What are the top selling vehicle models?",
        "arguments": ["date_range", "limit"],
        "source_table": SOURCE_TABLE
    },
    {
        "function_name": "get_sales_by_rep",
        "business_question": "Which sales representatives sold the most?",
        "arguments": ["date_range"],
        "source_table": SOURCE_TABLE
    },
    {
        "function_name": "get_revenue_by_state",
        "business_question": "How much revenue was generated by state?",
        "arguments": ["date_range"],
        "source_table": SOURCE_TABLE
    },
    {
        "function_name": "get_average_sale_price",
        "business_question": "What is the average sale price for a vehicle model?",
        "arguments": ["product_model", "date_range"],
        "source_table": SOURCE_TABLE
    },
    {
        "function_name": "get_dealership_sales_summary",
        "business_question": "How did each dealership perform?",
        "arguments": ["date_range"],
        "source_table": SOURCE_TABLE
    }
]

functions_df = spark.createDataFrame(validated_functions)

display(functions_df)

arguments,business_question,function_name,source_table
"List(product_model, date_range)",How many units and revenue for one vehicle model?,get_product_sales_summary,auto_sales_transactions
"List(date_range, limit)",What are the top selling vehicle models?,get_top_selling_models,auto_sales_transactions
List(date_range),Which sales representatives sold the most?,get_sales_by_rep,auto_sales_transactions
List(date_range),How much revenue was generated by state?,get_revenue_by_state,auto_sales_transactions
"List(product_model, date_range)",What is the average sale price for a vehicle model?,get_average_sale_price,auto_sales_transactions
List(date_range),How did each dealership perform?,get_dealership_sales_summary,auto_sales_transactions


In [0]:
# Save function candidates as delta table
FUNCTION_CANDIDATES_TABLE = "assistant_function_candidates"

functions_df.write.format("delta").mode("overwrite").saveAsTable(FUNCTION_CANDIDATES_TABLE)

print(f"Function candidates table created: {FUNCTION_CANDIDATES_TABLE}")

Function candidates table created: assistant_function_candidates
